# 13.11 Implementasi Python
## Praktikum 13.1 Persiapan Library

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## Praktikum 13.2 Import Dataset

Jika folder `dataset/train` dan `dataset/val` **ada**, dataset itu yang dipakai (struktur: `dataset/train/kelas_1`, `dataset/train/kelas_2`, dst.). Jika **tidak ada**, otomatis memakai dataset bunga (*flower_photos*, 5 kelas) yang diunduh dari Google.

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
SEED = 42

if os.path.isdir("dataset/train") and os.path.isdir("dataset/val"):
    print("Memakai dataset lokal")
    train_ds = keras.utils.image_dataset_from_directory(
        "dataset/train", image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED)
    val_ds = keras.utils.image_dataset_from_directory(
        "dataset/val", image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)
else:
    print("Folder dataset/ tidak ditemukan -> memakai flower_photos")
    url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
    data_dir = keras.utils.get_file("flower_photos", origin=url, untar=True)
    data_dir = os.path.join(os.path.dirname(data_dir), "flower_photos") \
        if not os.path.isdir(os.path.join(data_dir, "daisy")) else data_dir
    train_ds = keras.utils.image_dataset_from_directory(
        data_dir, validation_split=0.2, subset="training", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE)
    val_ds = keras.utils.image_dataset_from_directory(
        data_dir, validation_split=0.2, subset="validation", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

# Simpan SEBELUM .map(), karena atribut class_names hilang setelah .map()
class_names = train_ds.class_names
jumlah_kelas = len(class_names)
print("Kelas:", class_names)

## Praktikum 13.3 Preprocessing

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (train_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
            .cache().shuffle(1000).prefetch(AUTOTUNE))
val_ds = (val_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
          .cache().prefetch(AUTOTUNE))

## Praktikum 13.4 Training Model (Transfer Learning dengan MobileNetV2)

In [ ]:
base_model = MobileNetV2(input_shape=IMG_SIZE + (3,),
                         include_top=False, weights="imagenet")
base_model.trainable = False  # Feature extraction: bekukan bobot pre-trained

# Augmentasi hanya aktif saat training (otomatis nonaktif saat evaluasi)
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),
])

model = keras.Sequential([
    keras.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    base_model,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(jumlah_kelas, activation="softmax"),
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

history = model.fit(train_ds, validation_data=val_ds, epochs=15)

## Praktikum 13.5 Evaluasi

In [ ]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"Akurasi Validasi: {val_acc:.3f}")

## Praktikum 13.6 Visualisasi

In [ ]:
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Akurasi")
plt.title("Kurva Akurasi - Transfer Learning MobileNetV2")
plt.legend()
plt.show()

## 13.12 Latihan Praktikum

Latih pada dataset CIFAR-10 (`keras.datasets.cifar10`) dan bandingkan dengan hasil Transfer Learning.

### Langkah 1: Load data CIFAR-10

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

# Normalisasi 0-1 khusus untuk CNN dari nol
x_train_n = x_train.astype("float32") / 255.0
x_test_n = x_test.astype("float32") / 255.0

### Langkah 2: CNN dari nol (32x32)

In [ ]:
model_cnn = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    keras.layers.Conv2D(32, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
model_cnn.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

hist_cnn = model_cnn.fit(x_train_n, y_train, epochs=15, batch_size=64,
                         validation_data=(x_test_n, y_test))
cnn_loss, cnn_acc = model_cnn.evaluate(x_test_n, y_test, verbose=0)
print(f"Akurasi Test CNN: {cnn_acc:.3f}")

### Langkah 3: Transfer Learning MobileNetV2 pada CIFAR-10

Gambar di-resize ke 96x96 (ukuran minimum yang didukung bobot ImageNet MobileNetV2) dan memakai `preprocess_input` pada data mentah 0-255. Disarankan memakai GPU (misalnya Google Colab).

In [ ]:
TL_SIZE = (96, 96)

def siapkan(x, y, training):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(10000, seed=SEED)
    ds = ds.batch(64)
    ds = ds.map(lambda a, b: (preprocess_input(tf.image.resize(tf.cast(a, tf.float32), TL_SIZE)), b),
                num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_c10 = siapkan(x_train, y_train, True)
test_c10 = siapkan(x_test, y_test, False)

base_c10 = MobileNetV2(input_shape=TL_SIZE + (3,), include_top=False, weights="imagenet")
base_c10.trainable = False

model_tl = keras.Sequential([
    keras.Input(shape=TL_SIZE + (3,)),
    base_c10,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation="softmax"),
])
model_tl.compile(optimizer="adam",
                 loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])

hist_tl = model_tl.fit(train_c10, validation_data=test_c10, epochs=5)
tl_loss, tl_acc = model_tl.evaluate(test_c10, verbose=0)
print(f"Akurasi Test Transfer Learning: {tl_acc:.3f}")

### Langkah 4: Perbandingan

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(hist_cnn.history["val_accuracy"], label="CNN dari nol")
ax[0].plot(hist_tl.history["val_accuracy"], label="Transfer Learning (MobileNetV2)")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Akurasi Validasi")
ax[0].set_title("Akurasi Validasi per Epoch"); ax[0].legend()

ax[1].bar(["CNN dari nol", "Transfer Learning"], [cnn_acc, tl_acc], color=["tab:blue", "tab:orange"])
ax[1].set_ylim(0, 1); ax[1].set_ylabel("Akurasi Test")
ax[1].set_title("Akurasi Akhir CIFAR-10")
for i, v in enumerate([cnn_acc, tl_acc]):
    ax[1].text(i, v + 0.01, f"{v:.3f}", ha="center")

plt.tight_layout(); plt.show()

print(f"{'Model':<28}{'Parameter':>12}{'Akurasi Test':>15}")
print(f"{'CNN dari nol':<28}{model_cnn.count_params():>12,}{cnn_acc:>15.3f}")
print(f"{'Transfer Learning':<28}{model_tl.count_params():>12,}{tl_acc:>15.3f}")

### Analisis

Isi sendiri setelah menjalankan notebook, misalnya:
- Model mana yang akurasinya lebih tinggi, dan selisihnya berapa?
- Mana yang lebih cepat konvergen (lihat kurva)?
- Mengapa transfer learning bisa unggul walaupun bobot dasarnya dilatih pada ImageNet dan bukan CIFAR-10?
- Apakah ada tanda overfitting (selisih training vs validasi)?
- Apa yang bisa dicoba selanjutnya (fine-tuning beberapa layer atas, ukuran gambar lebih besar, augmentasi)?